# Tennis Power Preprocess

## 1. Setup and locate the Tennis Power data

### 1.1 Define paths and collect Power files

In [1]:
from pathlib import Path
import polars as pl

MINI_PROJECT_ROOT = Path.cwd().parents[2]

DATA_ROOT = (
    MINI_PROJECT_ROOT
    / "Tennis Schema"
    / "tennis_data"
)

EXTRACT_ROOT = DATA_ROOT / "extracted"

power_files = sorted(
    EXTRACT_ROOT.glob("*/raw_tennis_power_parquet/*.parquet")
)

print("Power file appearances:", len(power_files))
print("First file:", power_files[0] if power_files else "No files found")

Power file appearances: 22724
First file: d:\Learning\Daneshkar\Statistics\Stat with Python\Mini Project\Tennis Schema\tennis_data\extracted\20240201\raw_tennis_power_parquet\power_11998445.parquet


## 2. Inspect the Tennis Power data structure

### 2.1 Inspect one sample Power file

In [3]:
power_sample = pl.read_parquet(power_files[0])

print("Shape:", power_sample.shape)
print("Schema:", power_sample.schema)

power_sample.head(10)

Shape: (33, 5)
Schema: Schema({'match_id': Int64, 'set_num': Int64, 'game_num': Int64, 'value': Float64, 'break_occurred': Boolean})


match_id,set_num,game_num,value,break_occurred
i64,i64,i64,f64,bool
11998445,1,1,-52.8,false
11998445,1,2,48.14,false
11998445,1,3,-51.62,false
11998445,1,4,10.0,false
11998445,1,5,26.6,true
11998445,1,6,10.0,false
11998445,1,7,-10.0,false
11998445,1,8,-59.3,true
11998445,1,9,-33.02,false


## 3. Check essential data quality issues

### 3.1 Check schema consistency and missing values

In [5]:
schemas = [pl.read_parquet_schema(file) for file in power_files]

print("Unique schemas:", len({tuple(schema.items()) for schema in schemas}))

for column in power_sample.columns:
    dtypes = {schema[column] for schema in schemas}
    if len(dtypes) > 1:
        print(column, dtypes)

Unique schemas: 2
value {Int64, Float64}


### 3.2 Standardize value type and check missing values

In [6]:
power_all = pl.concat([
    pl.read_parquet(file).with_columns(
        pl.col("value").cast(pl.Float64)
    )
    for file in power_files
])

print("Shape:", power_all.shape)
power_all.null_count()

Shape: (469677, 5)


match_id,set_num,game_num,value,break_occurred
u32,u32,u32,u32,u32
0,0,0,0,0


## 4. Build and clean the full Tennis Power dataset

### 4.1 Build the full Power dataset with snapshot date

In [7]:
power_all = pl.concat([
    pl.read_parquet(file)
    .with_columns(
        pl.col("value").cast(pl.Float64),
        pl.lit(file.parents[1].name)
        .str.strptime(pl.Date, "%Y%m%d")
        .alias("snapshot_date")
    )
    for file in power_files
])

print("Shape:", power_all.shape)
print("Date range:", power_all["snapshot_date"].min(), "to", power_all["snapshot_date"].max())

Shape: (469677, 6)
Date range: 2024-02-01 to 2024-03-31


## 5. Validate the cleaned dataset

### 5.1 Check for duplicate Power records

In [8]:
duplicate_power = (
    power_all
    .group_by(["snapshot_date", "match_id", "set_num", "game_num"])
    .len()
    .filter(pl.col("len") > 1)
)

print("Duplicate Power keys:", duplicate_power.height)

Duplicate Power keys: 0


### 5.2 Check basic invalid values

In [9]:
power_all.select(
    (pl.col("set_num") < 1).sum().alias("invalid_set_num"),
    (pl.col("game_num") < 1).sum().alias("invalid_game_num"),
    (pl.col("value").is_nan() | pl.col("value").is_infinite()).sum().alias("invalid_value"),
)

invalid_set_num,invalid_game_num,invalid_value
u32,u32,u32
0,0,0


## 6. Save the processed data

### 6.1 Save the processed Power data

In [10]:
PROCESSED_ROOT = DATA_ROOT / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

power_output = PROCESSED_ROOT / "tennis_power_clean.parquet"

power_all.write_parquet(power_output)

saved_power = pl.read_parquet(power_output)

print("Saved shape:", saved_power.shape)
print("Exact match:", saved_power.equals(power_all))

Saved shape: (469677, 6)
Exact match: True


## Tennis Power Preprocessing Summary

- Found **22,724** Tennis Power file appearances.
- The files had **2 schema variations**, caused only by the `value` column being stored as either `Int64` or `Float64`.
- Standardized `value` to **Float64** so all files could be combined consistently.
- No missing values were found in any of the five original columns.
- Combined all daily snapshots into one dataset and added `snapshot_date`.
- Final dataset shape: **469,677 rows × 6 columns**.
- Snapshot dates range from **2024-02-01 to 2024-03-31**.
- No duplicate records were found for the key:
  `snapshot_date + match_id + set_num + game_num`.
- No invalid set numbers, game numbers, NaN values, or infinite Power values were found.
- No rows needed to be removed.
- Saved the cleaned dataset as `processed/tennis_power_clean.parquet`.
- The saved file was read back successfully and matched the processed DataFrame exactly.